In [ ]:
import os
import numpy as np
import pandas as pd
import gc

# Models from the following notebooks are used to create an ensemble
* LB: 0.0217 - https://www.kaggle.com/tarique7/hnm-exponential-decay-with-alternate-items/notebook
* LB: 0.0220 - https://www.kaggle.com/code/hengzheng/time-is-our-best-friend-v2/notebook
* LB: 0.0221 - https://www.kaggle.com/astrung/lstm-sequential-modelwith-item-features-tutorial
* LB: 0.0224 - https://www.kaggle.com/code/hirotakanogami/h-m-eda-customer-clustering-by-kmeans
* LB: 0.0225 - https://www.kaggle.com/lunapandachan/h-m-trending-products-weekly-add-test/notebook
* LB: 0.0227 - https://www.kaggle.com/code/hechtjp/h-m-eda-rule-base-by-customer-age
* LB: 0.0231 - https://www.kaggle.com/code/ebn7amdi/trending/notebook?scriptVersionId=90980162

In [ ]:
sub2 = pd.read_csv('../input/handmbestperforming/hnm-exponential-decay-with-alternate-items.csv').sort_values('customer_id').reset_index(drop=True)
sub5 = pd.read_csv('../input/handmbestperforming/time-is-our-best-friend-v2.csv').sort_values('customer_id').reset_index(drop=True)
sub3 = pd.read_csv('../input/handmbestperforming/lstm-sequential-modelwith-item-features-tutorial.csv').sort_values('customer_id').reset_index(drop=True)
sub4 = pd.read_csv('../input/hm-00224-solution/submission.csv').sort_values('customer_id').reset_index(drop=True)

sub0 = pd.read_csv('../input/hm-00231-solution/submission.csv').sort_values('customer_id').reset_index(drop=True)
sub1 = pd.read_csv('../input/handmbestperforming/h-m-trending-products-weekly-add-test.csv').sort_values('customer_id').reset_index(drop=True)
sub6 = pd.read_csv('../input/handmbestperforming/rule-based-by-customer-age.csv').sort_values('customer_id').reset_index(drop=True)

In [ ]:
sub0.columns = ['customer_id', 'prediction0']
sub0['prediction1'] = sub1['prediction']
sub0['prediction2'] = sub2['prediction']
sub0['prediction3'] = sub3['prediction']
sub0['prediction4'] = sub4['prediction']
sub0['prediction5'] = sub5['prediction']
sub0['prediction6'] = sub6['prediction']

del sub1, sub2, sub3, sub4, sub5, sub6
gc.collect()
sub0.head()

In [ ]:
def cust_blend2(dt, weights):
    REC = []
    REC.append(dt['prediction0'].str.split())
    REC.append(dt['prediction1'].str.split())
    REC.append(dt['prediction2'].str.split())
    REC.append(dt['prediction3'].str.split())
    REC.append(dt['prediction4'].str.split())
    REC.append(dt['prediction5'].str.split())
    REC.append(dt['prediction6'].str.split())
    
    res = {}
    W = weights
    
    for M in range(len(REC)):
        for custom_cnt, rec_list in enumerate(REC[M]):
            if custom_cnt not in res:
                res[custom_cnt]={}
                for idx, item in enumerate(rec_list):
                    if item in res[custom_cnt]:
                        res[custom_cnt][item] += W[M]/(idx+1)
                    else:
                        res[custom_cnt][item] = W[M]/(idx+1)
    
    for custom_cnt in range(len(res)):
        res[custom_cnt] = ' '.join(list(dict(sorted(res[custom_cnt].items(), key=lambda item: -item[1])).keys())[:12])
    
    return res
#     return pd.DataFrame.from_dict(res)

In [ ]:
w = np.array([1.05,1.00,0.95,0.85,0.75,0.95,0.55]) 
result = cust_blend2(sub0, w)

In [ ]:
import operator
for k,v in sorted(result.items(), key=operator.itemgetter(1))[:50]:
    print (k,v)

In [ ]:
sub0['prediction'] = np.array(list(result.values()))

sub0.head()

# Make a submission

In [ ]:
del sub0['prediction0']
del sub0['prediction1']
del sub0['prediction2']
del sub0['prediction3']
del sub0['prediction4']
del sub0['prediction5']
del sub0['prediction6']
gc.collect()


sub0.to_csv('submission.csv', index=False)